In [52]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import geopandas as gpd
import plotly.express as px
from unidecode import unidecode
from gurobipy import Model, GRB, quicksum

In [53]:
# ------------------ DADOS DE ENTRADA ------------------ #

# Conjunto de distritos
I = list(range(1, 97))
J = I

# Capacidades dos Núcleos
Cap = {j: 2300 for j in J}

# Matriz de distâncias dos distritos
matriz_df = pd.read_csv("matriz_distancias_distritos.csv", index_col=0)

# Padronizar nomes da matriz (maiúsculas, sem acento)
matriz_df.index = (
    matriz_df.index.str.upper()
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("ascii")
)

# Lista de distritos na ordem da matriz
distritos_sp = list(matriz_df.index)

# Ler CSV da Demanda PopRua
demanda_df = pd.read_csv("demanda_por_distrito.csv", sep=";")

# Padronizar nomes dos distritos no CSV da Demanda PopRua
demanda_df["distrito"] = (
    demanda_df["distrito"].str.upper()
    .str.normalize("NFKD")
    .str.encode("ascii", "ignore")
    .str.decode("ascii")
)

demanda_df = demanda_df.set_index("distrito")

# Padronização dos nomes do distrito São Miguel Paulista -> São Miguel
demanda_df = demanda_df.rename(index={"SAO MIGUEL PAULISTA": "SAO MIGUEL"})

# Aferição dos distritos sobre a necessidade da demanda
missing = set(distritos_sp) - set(demanda_df.index)
if missing:
    raise ValueError(f"Distritos sem demanda no CSV: {missing}")

# Reordenamento da demanda na mesma ordem de distritos_sp
demanda_df = demanda_df.loc[distritos_sp]

# Demanda D[i]
D = {i: float(demanda_df.iloc[i-1]["demanda"]) for i in I}

# Matriz de distâncias dist_ij
dist = {i: {} for i in I}
for i_idx, i in enumerate(I):
    distrito_i = distritos_sp[i_idx]
    for j_idx, j in enumerate(J):
        distrito_j = distritos_sp[j_idx]
        dist[i][j] = float(matriz_df.loc[distrito_i, distrito_j])

In [54]:
# ------------------ CRIAR MODELO NO GUROBI ------------------ #

m = Model("Localizacao_14_Nucleos")

# ------------------ VARIÁVEIS DE DECISÃO ------------------ #

# PopRua do distrito i atendido no Núcleo j

x = m.addVars(I, J, vtype=GRB.INTEGER, name="x")

# y[j] = 1 se Núcleo j é aberto
y = m.addVars(J, vtype=GRB.BINARY, name="y")

m.update()

In [55]:
# ------------------ FUNÇÃO OBJETIVO ------------------ #

# Minimizar a soma das distâncias

m.setObjective(
    quicksum(dist[i][j] * x[i, j] for i in I for j in J),
    GRB.MINIMIZE
)


# ------------------ RESTRIÇÕES ------------------ #

# (1) Atender demanda de cada distrito i
m.addConstrs(
    (quicksum(x[i, j] for j in J) == D[i] for i in I),
    name="demanda_distritos"
)

# (2) Capacidade das vagas dos Núcleos de Convivência

m.addConstrs(
    (quicksum(x[i, j] for i in I) <= Cap[j] * y[j] for j in J),
    name="capacidade_nucleos"
)

# (3) Abrir pelo menos 1 e no máximo 14 Núcleos

m.addConstr(
    quicksum(y[j] for j in J) == 14,
    name="qtd_nucleos_14"
)

# ------------------ OTIMIZAÇÃO ------------------ #

m.optimize()

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i3-8130U CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 193 rows, 9312 columns and 18624 nonzeros (Min)
Model fingerprint: 0x906d6efd
Model has 9120 linear objective coefficients
Variable types: 0 continuous, 9312 integer (96 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [2e+03, 8e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+03]
Found heuristic solution: objective 6.139215e+08
Presolve removed 2 rows and 192 columns
Presolve time: 0.10s
Presolved: 191 rows, 9120 columns, 18240 nonzeros
Variable types: 0 continuous, 9120 integer (192 binary)

Root relaxation: objective 6.765054e+06, 196 iterations, 0.03 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl | 

In [56]:
if m.Status != GRB.OPTIMAL:
    print(f"Modelo não está ótimo. Status = {m.Status}")
else:
    labels = []
    sources = []
    targets = []
    values = []
    node_index = {}

    # distritos
    for i in I:
        nome_dist = distritos_sp[i-1]
        node_index[("D", i)] = len(labels)
        labels.append(f"Distrito {nome_dist}")

    # todos os núcleos 1..14
    for j in J:
        node_index[("N", j)] = len(labels)
        labels.append(f"Núcleo {j}")

    # fluxos positivos
    for i in I:
        for j in J:
            flow = x[i, j].X
            if flow > 1e-6:
                sources.append(node_index[("D", i)])
                targets.append(node_index[("N", j)])
                values.append(flow)

    print("Qtde de links no Sankey:", len(values))
    if len(values) == 0:
        print("Nenhum fluxo x[i,j] > 0 na solução.")
    else:
        fig = go.Figure(data=[go.Sankey(
            node=dict(
                pad=20,
                thickness=25,
                line=dict(color="black", width=0.5),
                label=labels
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values,
                color="rgba(0, 100, 200, 0.5)"
            )
        )])

        fig.update_layout(
            title_text=" Abertura para 14 Núcleos de Convivência",
            font_size=12
        )
        fig.write_html("sankey_localizacao.html", auto_open=True)


Qtde de links no Sankey: 105
